# 🔭 LLM Cost Observatory — Finding Where Your LLM Spend Actually Goes

**The problem:** Most teams treat LLM cost as a pricing question. It isn't — it's an *architecture* question. In a multi-turn agentic system, the same conversation gets re-sent to the model on every single turn. By turn 10, you're paying for turns 1–9 again. And again. And again.

This notebook builds a **diagnostic system** that:

1. Detects **context bloat** — input tokens growing faster than the conversation actually requires
2. Fingerprints the **root cause** — history accumulation vs. tool output injection vs. RAG over-fetch
3. Quantifies **dollar impact** per problem, per workflow
4. Generates **prioritized fixes** ranked by savings ÷ effort
5. Flags **regressions** — cost spikes and cache degradation over time

We'll compare two synthetic systems: a well-optimised one and a broken one, then show exactly how much money the broken one is leaving on the table.

> Full project: [github.com/chowdarymcs/llm-cost-observatory](https://github.com/chowdarymcs/llm-cost-observatory)

---
## 1. Setup

We generate synthetic Langfuse-style observability traces. Each record represents one LLM generation call with token counts, model, timestamp, and session grouping.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette("husl")
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

print("Ready.")

---
## 2. Model Pricing

Cost is driven by three token types. The critical detail most people miss: **cache reads cost ~10× less than regular input tokens**. If your system prompt is 2,000 tokens and you're not caching it, you're paying full price for it on every single call.

In [ ]:
# USD per 1M tokens
MODEL_PRICING = {
    "claude-opus-4-6":   {"input": 15.00, "output": 75.00, "cache_read": 1.50},
    "claude-sonnet-4-6": {"input":  3.00, "output": 15.00, "cache_read": 0.30},
    "claude-haiku-4-5":  {"input":  0.80, "output":  4.00, "cache_read": 0.08},
    "gpt-4o":            {"input":  2.50, "output": 10.00, "cache_read": 1.25},
    "gpt-4o-mini":       {"input":  0.15, "output":  0.60, "cache_read": 0.075},
}
DEFAULT_PRICING = {"input": 2.00, "output": 8.00, "cache_read": 0.50}

pricing_df = pd.DataFrame(MODEL_PRICING).T
pricing_df['input_vs_cache'] = (pricing_df['input'] / pricing_df['cache_read']).round(1)
print("Cost per 1M tokens:")
display(pricing_df)
print(f"\nCaching is {pricing_df['input_vs_cache'].mean():.0f}x cheaper than re-sending input tokens.")

---
## 3. Generating Realistic Trace Data

Two scenarios:

| | **Optimised** | **Unoptimised** |
|---|---|---|
| History handling | Rolling summarization | Full re-injection every turn |
| Tool outputs | Compressed before injection | Raw payloads dumped in |
| RAG | Two-phase (anchor → fetch) | Full chunks always |
| Caching | Enabled on static content | Not configured |
| Model routing | Task-appropriate | Opus for trivial classification |

The key modelling insight is in how `input_tokens` grows across turns within a session.

In [ ]:
# (workflow, model, avg_turns, base_input, base_output, sessions/day, cache_ratio, bloat_factor)
OPTIMISED = [
    ("document-qa/retrieve",   "claude-haiku-4-5",  3, 1200,  180, 40, 0.72, 1.0),
    ("document-qa/generate",   "claude-sonnet-4-6", 4, 2200,  650, 35, 0.68, 1.0),
    ("support-agent/classify", "claude-haiku-4-5",  1,  600,   90, 90, 0.80, 1.0),
    ("support-agent/respond",  "claude-sonnet-4-6", 5, 1800,  520, 55, 0.65, 1.0),
    ("data-extraction/parse",  "claude-haiku-4-5",  2,  900,  240, 70, 0.75, 1.0),
    ("report-writer/draft",    "claude-sonnet-4-6", 6, 2800, 1400, 12, 0.60, 1.0),
]

UNOPTIMISED = [
    ("document-qa/retrieve",       "claude-haiku-4-5",  3, 1200, 180,  40, 0.70, 1.0),
    ("document-qa/generate",       "claude-sonnet-4-6", 4, 2200, 650,  35, 0.62, 1.0),
    # PROBLEM 1: full history re-injection on every turn
    ("research-agent/investigate", "claude-sonnet-4-6",14, 3000, 700,  18, 0.05, 6.5),
    # PROBLEM 2: raw tool outputs dumped into context
    ("data-pipeline/execute",      "claude-sonnet-4-6", 8, 4500, 400,  25, 0.10, 2.8),
    # PROBLEM 3: RAG fetches full documents regardless of query
    ("knowledge-base/search",      "claude-sonnet-4-6", 2, 9500, 350,  30, 0.08, 1.0),
    # PROBLEM 4: premium model doing trivial classification
    ("triage/categorize",          "claude-opus-4-6",   1,  800,  85, 120, 0.12, 1.0),
    ("support-agent/classify",     "claude-haiku-4-5",  1,  600,  90,  90, 0.55, 1.0),
]

print(f"Optimised:   {len(OPTIMISED)} workflows")
print(f"Unoptimised: {len(UNOPTIMISED)} workflows (4 with injected problems)")

In [ ]:
def generate_traces(workflows, days=30, seed=42, inject_regressions=False):
    """
    Generate synthetic LLM observability traces.

    The core mechanic: in a healthy session, input_tokens for turn N should be
    roughly base_input + (sum of prior outputs). When bloat_factor > 1, we
    simulate re-injecting the full history repeatedly, so input grows far faster.
    """
    rng = np.random.default_rng(seed)
    end_date = datetime.utcnow()
    start_date = end_date - timedelta(days=days)
    midpoint = start_date + timedelta(days=days/2)

    rows, counter = [], 0

    for name, model, avg_turns, base_in, base_out, spd, cache_ratio, bloat in workflows:
        for s in range(int(spd * days)):
            session_id = f"sess_{name.split('/')[0][:6]}_{s:05d}"
            session_start = start_date + timedelta(hours=rng.uniform(0, days*24))
            is_late = session_start > midpoint

            # Inject a traffic/payload spike in the second half for one workflow
            spike = 3.2 if (inject_regressions and name == "data-pipeline/execute" and is_late) else 1.0

            n_turns = max(1, int(rng.normal(avg_turns, avg_turns*0.3)))
            trace_id, cum_output = f"trace_{counter:07d}", 0

            for turn in range(n_turns):
                counter += 1
                ts = session_start + timedelta(seconds=turn*rng.uniform(8, 45))

                if bloat > 1.0:
                    # Bloated: prior outputs re-injected multiple times over
                    input_tok = (base_in + cum_output*bloat) * rng.normal(1.0, 0.15) * spike
                    # Tool output spikes on random turns
                    if name == "data-pipeline/execute" and rng.random() < 0.35:
                        input_tok *= rng.uniform(1.8, 3.5)
                else:
                    # Healthy: input grows only by prior output
                    input_tok = (base_in + cum_output*0.85) * rng.normal(1.0, 0.10)

                input_tok = max(100, input_tok)
                output_tok = max(20, base_out * rng.normal(1.0, 0.25))
                cum_output += output_tok

                eff_cache = cache_ratio * (0.45 if (inject_regressions and is_late) else 1.0)
                cache_tok = input_tok * eff_cache
                billable  = input_tok * (1 - eff_cache)

                p = MODEL_PRICING.get(model, DEFAULT_PRICING)
                ic = billable  / 1e6 * p["input"]
                oc = output_tok / 1e6 * p["output"]
                cc = cache_tok  / 1e6 * p["cache_read"]

                rows.append({
                    "id": f"obs_{counter:07d}", "trace_id": trace_id,
                    "workflow": name, "session_id": session_id, "model": model,
                    "timestamp": ts, "turn_index": turn,
                    "input_tokens": int(billable), "output_tokens": int(output_tok),
                    "cache_read_tokens": int(cache_tok),
                    "input_cost": ic, "output_cost": oc, "cache_read_cost": cc,
                    "total_cost": ic + oc + cc,
                })

    df = pd.DataFrame(rows)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.sort_values("timestamp").reset_index(drop=True)


df_good = generate_traces(OPTIMISED,   days=30, inject_regressions=False)
df_bad  = generate_traces(UNOPTIMISED, days=30, inject_regressions=True)

print(f"Optimised system:   {len(df_good):,} observations, {df_good.session_id.nunique():,} sessions")
print(f"Unoptimised system: {len(df_bad):,} observations, {df_bad.session_id.nunique():,} sessions")

---
## 4. First Look — The Cost Gap

In [ ]:
def summarize(df, label):
    cache_hit = df.cache_read_tokens.sum() / (df.input_tokens.sum() + df.cache_read_tokens.sum())
    return {
        "System": label,
        "Total Cost": df.total_cost.sum(),
        "Observations": len(df),
        "Sessions": df.session_id.nunique(),
        "Avg Input Tokens": df.input_tokens.mean(),
        "Cache Hit Rate": cache_hit * 100,
        "Cost/Session": df.total_cost.sum() / df.session_id.nunique(),
    }

comparison = pd.DataFrame([summarize(df_good, "Optimised"), summarize(df_bad, "Unoptimised")])
display(comparison.set_index("System").T)

mult = df_bad.total_cost.sum() / df_good.total_cost.sum()
print(f"\nThe unoptimised system costs {mult:.1f}x more.")
print(f"Same workload class — the difference is entirely architectural.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Total cost
axes[0].bar(["Optimised", "Unoptimised"],
            [df_good.total_cost.sum(), df_bad.total_cost.sum()],
            color=["#2ECC71", "#E74C3C"])
axes[0].set_title("Total 30-Day Spend", fontweight="bold")
axes[0].set_ylabel("USD")
for i, v in enumerate([df_good.total_cost.sum(), df_bad.total_cost.sum()]):
    axes[0].text(i, v, f"${v:,.0f}", ha="center", va="bottom", fontweight="bold")

# Avg input tokens
axes[1].bar(["Optimised", "Unoptimised"],
            [df_good.input_tokens.mean(), df_bad.input_tokens.mean()],
            color=["#2ECC71", "#E74C3C"])
axes[1].set_title("Avg Billable Input Tokens per Call", fontweight="bold")
axes[1].set_ylabel("Tokens")
for i, v in enumerate([df_good.input_tokens.mean(), df_bad.input_tokens.mean()]):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom", fontweight="bold")

# Cache hit rate
ch_g = df_good.cache_read_tokens.sum()/(df_good.input_tokens.sum()+df_good.cache_read_tokens.sum())*100
ch_b = df_bad.cache_read_tokens.sum()/(df_bad.input_tokens.sum()+df_bad.cache_read_tokens.sum())*100
axes[2].bar(["Optimised", "Unoptimised"], [ch_g, ch_b], color=["#2ECC71", "#E74C3C"])
axes[2].set_title("Cache Hit Rate", fontweight="bold")
axes[2].set_ylabel("%")
for i, v in enumerate([ch_g, ch_b]):
    axes[2].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontweight="bold")

plt.tight_layout(); plt.show()

---
## 5. The Core Metric — Bloat Score

Here's the central idea of this whole analysis.

In a healthy multi-turn session, the input tokens for turn *N* should be approximately:

$$\text{expected input}_N = \text{system prompt} + \sum_{i=1}^{N-1} \text{output}_i$$

You re-send what was said before, plus your instructions. That's it.

$$\text{Bloat Score} = \frac{\text{actual input tokens}}{\text{expected input tokens}}$$

- **< 2×** — healthy
- **2–5×** — moderate waste
- **> 5×** — severe; something is being re-injected repeatedly

The score is model-agnostic and needs no instrumentation beyond token counts you already have.

In [ ]:
SYSTEM_PROMPT_ESTIMATE = 500

def compute_bloat_scores(df):
    """Per-session bloat score and estimated dollar waste."""
    out = []
    for sid, grp in df.groupby("session_id"):
        grp = grp.sort_values("turn_index")
        if len(grp) < 2:
            continue

        cum_output = grp.output_tokens.cumsum().shift(1).fillna(0)
        expected   = cum_output + SYSTEM_PROMPT_ESTIMATE

        last_row      = grp.iloc[-1]
        last_expected = expected.iloc[-1]
        bloat_score   = last_row.input_tokens / max(last_expected, 1)

        waste_tokens = max(0, int((grp.input_tokens - expected).clip(lower=0).sum()))
        model = grp.model.mode()[0]
        price = MODEL_PRICING.get(model, DEFAULT_PRICING)["input"]

        out.append({
            "session_id":  sid,
            "workflow":    grp.workflow.mode()[0],
            "turns":       len(grp),
            "bloat_score": round(bloat_score, 2),
            "waste_tokens": waste_tokens,
            "waste_usd":   waste_tokens / 1e6 * price,
            "severity":    "Severe" if bloat_score >= 5 else ("Moderate" if bloat_score >= 2 else "Healthy"),
        })
    return pd.DataFrame(out).sort_values("bloat_score", ascending=False)


bloat_good = compute_bloat_scores(df_good)
bloat_bad  = compute_bloat_scores(df_bad)

print("OPTIMISED SYSTEM")
print(bloat_good.severity.value_counts().to_string())
print(f"  Avg bloat score: {bloat_good.bloat_score.mean():.2f}x")
print(f"  Estimated waste: ${bloat_good.waste_usd.sum():,.2f}")

print("\nUNOPTIMISED SYSTEM")
print(bloat_bad.severity.value_counts().to_string())
print(f"  Avg bloat score: {bloat_bad.bloat_score.mean():.2f}x")
print(f"  Estimated waste: ${bloat_bad.waste_usd.sum():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
colors = {"Healthy": "#2ECC71", "Moderate": "#F39C12", "Severe": "#E74C3C"}

for ax, bdf, title in [(axes[0], bloat_good, "Optimised"), (axes[1], bloat_bad, "Unoptimised")]:
    for sev in ["Healthy", "Moderate", "Severe"]:
        subset = bdf[bdf.severity == sev]
        if len(subset):
            ax.hist(subset.bloat_score, bins=40, alpha=0.75, label=sev, color=colors[sev])
    ax.axvline(2, ls="--", c="#F39C12", lw=1.5)
    ax.axvline(5, ls="--", c="#E74C3C", lw=1.5)
    ax.set_title(f"{title} — Bloat Score Distribution", fontweight="bold")
    ax.set_xlabel("Bloat Score (x)"); ax.set_ylabel("Sessions"); ax.legend()

plt.tight_layout(); plt.show()

### Watching bloat compound turn by turn

This is the clearest visual of the problem. Pick the worst session and plot what *should* have happened against what actually did.

In [ ]:
worst_sid = bloat_bad.iloc[0].session_id
best_sid  = bloat_good[bloat_good.turns >= 5].iloc[-1].session_id

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

for ax, sid, df_src, title in [
    (axes[0], best_sid,  df_good, "Healthy session"),
    (axes[1], worst_sid, df_bad,  "Bloated session"),
]:
    s = df_src[df_src.session_id == sid].sort_values("turn_index")
    expected = s.output_tokens.cumsum().shift(1).fillna(0) + SYSTEM_PROMPT_ESTIMATE

    ax.plot(s.turn_index, s.input_tokens, "o-", c="#E74C3C", lw=2, label="Actual input")
    ax.plot(s.turn_index, expected, "s--", c="#2ECC71", lw=2, label="Expected (no bloat)")
    ax.fill_between(s.turn_index, expected, s.input_tokens,
                    where=(s.input_tokens > expected), alpha=0.25, color="#E74C3C",
                    label="Wasted tokens")

    score = s.input_tokens.iloc[-1] / max(expected.iloc[-1], 1)
    ax.set_title(f"{title} — bloat {score:.1f}x", fontweight="bold")
    ax.set_xlabel("Turn"); ax.set_ylabel("Input tokens"); ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print("Left: input grows only as the conversation grows.")
print("Right: input compounds — every turn re-pays for the entire history.")

---
## 6. Root Cause Fingerprinting

A bloat score tells you *that* something is wrong. It doesn't tell you *what*. Different causes need completely different fixes, so we fingerprint them by their statistical signature:

| Pattern | Signature | Fix |
|---|---|---|
| **History accumulation** | Input grows steadily with turn index | Rolling summarization |
| **Tool output injection** | High input variance *within* a session (spikes) | Compress tool results |
| **RAG over-fetch** | Uniformly high input from turn 1 | Two-phase retrieval |
| **Cache miss** | Low cache-read ratio across the board | Enable prompt caching |
| **Model over-spend** | Premium model, tiny outputs | Route to cheaper model |

In [ ]:
def fingerprint_workflows(df, bloat_df):
    """Classify each workflow by its dominant cost anti-pattern."""
    results = []
    for wf, grp in df.groupby("workflow"):
        wf_bloat = bloat_df[bloat_df.workflow == wf]
        avg_bloat = wf_bloat.bloat_score.mean() if len(wf_bloat) else 1.0

        # Within-session input variability
        cv = (grp.groupby("session_id").input_tokens.std() /
              grp.groupby("session_id").input_tokens.mean().replace(0, 1)).mean()

        first_turn_input = grp[grp.turn_index == 0].input_tokens.mean()
        cache_ratio = grp.cache_read_tokens.sum() / max(
            grp.input_tokens.sum() + grp.cache_read_tokens.sum(), 1)
        model = grp.model.mode()[0]
        avg_output = grp.output_tokens.mean()

        patterns = []
        if avg_bloat >= 2.0:                       patterns.append("HISTORY_ACCUMULATION")
        if cv > 0.5:                                patterns.append("TOOL_OUTPUT_INJECTION")
        if first_turn_input > 4000:                 patterns.append("RAG_OVERFETCH")
        if cache_ratio < 0.20:                      patterns.append("CACHE_MISS")
        if model in ("claude-opus-4-6", "gpt-4o") and avg_output < 400:
            patterns.append("EXPENSIVE_MODEL_OVERUSE")

        results.append({
            "workflow": wf, "model": model,
            "total_cost": grp.total_cost.sum(),
            "avg_bloat": round(avg_bloat, 2),
            "input_cv": round(cv, 2),
            "first_turn_input": int(first_turn_input),
            "cache_ratio": round(cache_ratio, 2),
            "avg_output": int(avg_output),
            "patterns": ", ".join(patterns) if patterns else "—",
        })
    return pd.DataFrame(results).sort_values("total_cost", ascending=False)


fp = fingerprint_workflows(df_bad, bloat_bad)
print("UNOPTIMISED SYSTEM — root cause fingerprints\n")
display(fp[["workflow","model","total_cost","avg_bloat","input_cv",
            "first_turn_input","cache_ratio","patterns"]])

---
## 7. Quantifying the Fix — Dollars, Not Adjectives

"You should compress your context" is useless advice. "Rolling summarization on `research-agent/investigate` saves $383/month for medium effort" is actionable.

Each detected pattern gets a savings estimate based on its measured waste and a conservative recovery factor.

In [ ]:
RECOVERY_FACTOR = {
    "HISTORY_ACCUMULATION":    0.70,   # summarization recovers ~70% of compounding waste
    "TOOL_OUTPUT_INJECTION":   0.50,   # compression recovers ~50% of payload tokens
    "RAG_OVERFETCH":           0.40,   # two-phase retrieval cuts ~40% of retrieval tokens
    "CACHE_MISS":              0.90,   # caching saves ~90% on static tokens
    "EXPENSIVE_MODEL_OVERUSE": 0.80,   # cheaper model is ~80% less
}
EFFORT = {
    "HISTORY_ACCUMULATION": "Medium", "TOOL_OUTPUT_INJECTION": "Low",
    "RAG_OVERFETCH": "Medium", "CACHE_MISS": "Low", "EXPENSIVE_MODEL_OVERUSE": "Low",
}
EFFORT_SCORE = {"Low": 1, "Medium": 2, "High": 3}
FIX_TITLE = {
    "HISTORY_ACCUMULATION":    "Rolling summarization",
    "TOOL_OUTPUT_INJECTION":   "Compress tool outputs",
    "RAG_OVERFETCH":           "Two-phase retrieval",
    "CACHE_MISS":              "Enable prompt caching",
    "EXPENSIVE_MODEL_OVERUSE": "Route to cheaper model",
}

def build_recommendations(df, bloat_df, fp_df, days=30):
    recs = []
    for _, row in fp_df.iterrows():
        if row.patterns == "—":
            continue
        wf = row.workflow
        wf_data = df[df.workflow == wf]
        wf_cost = wf_data.total_cost.sum()
        monthly_cost = wf_cost / days * 30

        for pattern in row.patterns.split(", "):
            # Attribute a share of workflow cost to this pattern, then apply recovery
            if pattern == "HISTORY_ACCUMULATION":
                share = min(0.6, (row.avg_bloat - 1) / row.avg_bloat)
            elif pattern == "TOOL_OUTPUT_INJECTION":
                share = min(0.35, row.input_cv * 0.4)
            elif pattern == "RAG_OVERFETCH":
                share = 0.35
            elif pattern == "CACHE_MISS":
                share = 0.15
            else:  # EXPENSIVE_MODEL_OVERUSE
                share = 0.75

            savings = monthly_cost * share * RECOVERY_FACTOR[pattern]
            eff = EFFORT[pattern]
            recs.append({
                "workflow": wf, "pattern": pattern, "fix": FIX_TITLE[pattern],
                "monthly_savings": savings, "effort": eff,
                "priority_score": savings / EFFORT_SCORE[eff],
            })
    return pd.DataFrame(recs).sort_values("priority_score", ascending=False).reset_index(drop=True)


recs = build_recommendations(df_bad, bloat_bad, fp)
recs.index = recs.index + 1
print(f"Detected {len(recs)} optimization opportunities\n")
display(recs.style.format({"monthly_savings": "${:,.2f}", "priority_score": "{:,.2f}"}))
print(f"\nTotal identified monthly savings: ${recs.monthly_savings.sum():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Savings by fix
top = recs.head(10)
bar_colors = {"Low": "#2ECC71", "Medium": "#F39C12", "High": "#E74C3C"}
axes[0].barh(range(len(top)), top.monthly_savings,
             color=[bar_colors[e] for e in top.effort])
axes[0].set_yticks(range(len(top)))
axes[0].set_yticklabels([f"{r.fix}\n({r.workflow[:22]})" for _, r in top.iterrows()], fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel("Monthly Savings (USD)")
axes[0].set_title("Top Opportunities by Savings", fontweight="bold")
for i, v in enumerate(top.monthly_savings):
    axes[0].text(v, i, f" ${v:,.0f}", va="center", fontsize=8, fontweight="bold")

# Priority matrix: savings vs effort
for eff in ["Low", "Medium", "High"]:
    sub = recs[recs.effort == eff]
    if len(sub):
        axes[1].scatter(sub.effort.map(EFFORT_SCORE), sub.monthly_savings,
                        s=sub.monthly_savings*0.6+80, alpha=0.65,
                        color=bar_colors[eff], label=eff, edgecolors="white", linewidth=1.5)
axes[1].set_xticks([1,2,3]); axes[1].set_xticklabels(["Low","Medium","High"])
axes[1].set_xlabel("Implementation Effort"); axes[1].set_ylabel("Monthly Savings (USD)")
axes[1].set_title("Priority Matrix — fix top-left first", fontweight="bold")
axes[1].legend(title="Effort")

plt.tight_layout(); plt.show()

---
## 8. Savings Forecast

Stacking the fixes, with a cap per fix so we don't double-count overlapping savings.

In [ ]:
DAYS = 30
current_monthly = df_bad.total_cost.sum() / DAYS * 30

running = current_monthly
steps = [("Current", current_monthly, "total")]
seen = set()
for _, r in recs.iterrows():
    if r.pattern in seen:
        continue
    seen.add(r.pattern)
    saving = min(r.monthly_savings, running * 0.45)   # cap to stay conservative
    running -= saving
    steps.append((r.fix, -saving, "saving"))
steps.append(("Optimised", running, "total"))

total_savings = current_monthly - running
pct = total_savings / current_monthly * 100

fig, ax = plt.subplots(figsize=(13, 5))
cum, x = current_monthly, 0
for label, val, kind in steps:
    if kind == "total":
        ax.bar(x, val, color="#4F8EF7", width=0.6)
        ax.text(x, val, f"${val:,.0f}", ha="center", va="bottom", fontweight="bold")
        cum = val
    else:
        ax.bar(x, abs(val), bottom=cum+val, color="#2ECC71", width=0.6)
        ax.text(x, cum+val+abs(val), f"-${abs(val):,.0f}", ha="center",
                va="bottom", fontsize=9, color="#14663a", fontweight="bold")
        cum += val
    x += 1

ax.set_xticks(range(len(steps)))
ax.set_xticklabels([s[0] for s in steps], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Monthly Cost (USD)")
ax.set_title(f"30-Day Savings Forecast — ${total_savings:,.0f}/mo ({pct:.0f}% reduction)",
             fontweight="bold", fontsize=13)
plt.tight_layout(); plt.show()

print(f"Current:   ${current_monthly:,.2f}/month")
print(f"Optimised: ${running:,.2f}/month")
print(f"Savings:   ${total_savings:,.2f}/month  ({pct:.1f}%)  =  ${total_savings*12:,.0f}/year")

---
## 9. Regression Detection

Optimising once isn't enough — systems drift. A refactor re-enables full history injection; a prompt change breaks cache keys; a new workflow ships without review.

We split the window in half and compare, so no manual thresholds are needed.

In [ ]:
def detect_alerts(df, bloat_df):
    alerts = []
    mid = df.timestamp.min() + (df.timestamp.max() - df.timestamp.min()) / 2
    early, late = df[df.timestamp <= mid], df[df.timestamp > mid]

    # Cost spikes per workflow
    e_cost = early.groupby("workflow").total_cost.sum()
    l_cost = late.groupby("workflow").total_cost.sum()
    for wf in e_cost.index.union(l_cost.index):
        e, l = e_cost.get(wf, 0), l_cost.get(wf, 0)
        if e > 0.01 and l / e >= 2.0:
            alerts.append({
                "severity": "Critical" if l/e >= 3 else "Warning",
                "type": "COST_SPIKE", "workflow": wf,
                "detail": f"Cost rose {l/e:.1f}x (${e:,.2f} -> ${l:,.2f})",
            })

    # Cache degradation
    def cache_rate(d):
        return d.cache_read_tokens.sum() / max(d.input_tokens.sum() + d.cache_read_tokens.sum(), 1) * 100
    ce, cl = cache_rate(early), cache_rate(late)
    if ce > 5 and cl < ce * 0.8:
        alerts.append({
            "severity": "Warning", "type": "CACHE_DEGRADATION", "workflow": "(global)",
            "detail": f"Cache hit rate fell {ce:.1f}% -> {cl:.1f}% ({cl-ce:+.1f}pp)",
        })

    # New expensive workflows
    new_wfs = set(late.workflow) - set(early.workflow)
    avg_cost = df.groupby("workflow").total_cost.sum().mean()
    for wf in new_wfs:
        c = late[late.workflow == wf].total_cost.sum()
        if c > avg_cost:
            alerts.append({
                "severity": "Info", "type": "NEW_EXPENSIVE_WORKFLOW", "workflow": wf,
                "detail": f"New workflow appeared costing ${c:,.2f} ({c/avg_cost:.1f}x avg)",
            })

    order = {"Critical": 0, "Warning": 1, "Info": 2}
    return pd.DataFrame(alerts).sort_values("severity", key=lambda s: s.map(order)) if alerts else pd.DataFrame()


print("OPTIMISED SYSTEM")
a_good = detect_alerts(df_good, bloat_good)
print("  No anomalies detected.\n" if a_good.empty else a_good.to_string(index=False))

print("UNOPTIMISED SYSTEM")
a_bad = detect_alerts(df_bad, bloat_bad)
display(a_bad)

In [ ]:
# Visualise the spike that triggered the alert
spike_wf = "data-pipeline/execute"
daily = (df_bad[df_bad.workflow == spike_wf]
         .assign(date=lambda d: d.timestamp.dt.date)
         .groupby("date").total_cost.sum().reset_index())

fig, ax = plt.subplots(figsize=(13, 4))
mid_date = daily.date.iloc[len(daily)//2]
colors = ["#4F8EF7" if d <= mid_date else "#E74C3C" for d in daily.date]
ax.bar(daily.date, daily.total_cost, color=colors)
ax.axvline(mid_date, ls="--", c="#333", lw=1.5, label="Period midpoint")
ax.set_title(f"Cost spike detected — {spike_wf}", fontweight="bold")
ax.set_ylabel("Daily Cost (USD)"); ax.legend()
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

first_half  = daily[daily.date <= mid_date].total_cost.mean()
second_half = daily[daily.date >  mid_date].total_cost.mean()
print(f"First half avg:  ${first_half:,.2f}/day")
print(f"Second half avg: ${second_half:,.2f}/day  ({second_half/first_half:.1f}x)")

---
## 10. The Fixes

Each detected pattern maps to a concrete implementation.

### Rolling summarization — for history accumulation
```python
SUMMARIZE_AFTER_TURNS = 8

async def compress_history(history, llm):
    if len(history) <= SUMMARIZE_AFTER_TURNS:
        return history
    old, recent = history[:-SUMMARIZE_AFTER_TURNS], history[-SUMMARIZE_AFTER_TURNS:]
    summary = await llm.complete(f"Summarize in 150 words:\n{old}")
    return [{"role": "assistant", "content": f"[Summary]: {summary}"}] + recent
```

### Tool output compression — for injection spikes
```python
MAX_TOOL_TOKENS = 400

def compress_tool_output(raw, max_chars=MAX_TOOL_TOKENS * 4):
    if isinstance(raw, dict):
        raw = json.dumps(raw, indent=2)
    if len(raw) <= max_chars:
        return raw
    lines = raw.strip().split("\n")
    return f"{chr(10).join(lines[:15])}\n... [{len(lines)-20} lines elided] ...\n{chr(10).join(lines[-5:])}"
```

### Two-phase retrieval — for RAG over-fetch
```python
async def two_phase_retrieve(query, store, k_anchor=5):
    # Phase 1: lightweight anchors (id + summary only)
    anchors = await store.search(query, k=k_anchor, fields=["id", "summary"])
    # Phase 2: full content for the best match only
    full = await store.fetch(anchors[0]["id"])
    return anchors, full
```

### Prompt caching — for cache misses
```python
response = client.messages.create(
    model="claude-sonnet-4-6",
    system=[{
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"},   # <- the whole fix
    }],
    messages=history,
)
```

### Model routing — for over-spend
```python
def route_model(task_type, est_output_tokens):
    if task_type in ("classification", "extraction", "summarization"):
        return "claude-haiku-4-5"
    if est_output_tokens < 500:   return "claude-haiku-4-5"
    if est_output_tokens < 2000:  return "claude-sonnet-4-6"
    return "claude-opus-4-6"
```

---
## 11. Takeaways

**What this analysis found in the unoptimised system:**

1. **Context bloat is the dominant cost driver.** Sessions averaging 5×+ bloat scores meant most spend went to re-processing conversation history, not generating new output.
2. **Cache configuration is the cheapest win available.** Cache reads cost ~10× less than input tokens. A 6% hit rate versus a 66% hit rate is pure architecture, not pricing.
3. **Model routing matters more than model choice.** Running a premium model for single-turn classification with 85-token outputs is a routing bug, not a capability requirement.
4. **Cost distribution is never uniform.** A handful of workflows drove the majority of spend — averages hide exactly the sessions you need to find.
5. **Regressions are inevitable.** A system optimised once will drift. Detection needs to be continuous and baseline-relative, not threshold-based.

**Method summary:** the bloat score works because it needs nothing beyond token counts you're already logging. No code instrumentation, no tracing changes — just a ratio against what the conversation mathematically required.

---

The full implementation — with a Streamlit dashboard, Langfuse/ClickHouse connectors, and an HTML report generator — is on GitHub:

**[github.com/chowdarymcs/llm-cost-observatory](https://github.com/chowdarymcs/llm-cost-observatory)**

If this was useful, an upvote helps others find it. 🙏